In [1]:
# Load Packages 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from functools import reduce
from plotnine import * 
import statsmodels.formula.api as smf 
from pathlib import Path

In [2]:
# Merge Diagnostics Helper Function 
def merge_with_diagnostics(left, right, on, how, name, validate=None, suffixes=("_x", "_y")):
    before_left = len(left)
    before_right = len(right)

    merged = left.merge(
        right,
        on=on,
        how=how,
        indicator=True,
        validate=validate,
        suffixes=suffixes
    )

    print(f"\n{name}")
    print(f"Left rows before merge: {before_left}")
    print(f"Right rows before merge: {before_right}")
    print(f"Rows after merge: {len(merged)}")
    print(merged["_merge"].value_counts())

    return merged.drop(columns="_merge")

In [3]:
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "code"/"data"
OUTPUT_DIR = PROJECT_DIR / "output"

OUTPUT_DIR.mkdir(exist_ok=True)

In [4]:
# Load CAS dataset 

cas_df = pd.read_csv(DATA_DIR/"cas_data.csv") 
cas_df 

,id,Data rilevazione...2,id_centro,Nome centro,original_name,Indirizzo centro,comune_id,Comune,comune_url,Classe comune,...,Prezzo pro die pro capite,Presenze giornaliere,Capienza,capienza_no_effettiva,old_capienza,indicatore_capienza,Procedeura,operativita,Tipologia centro,Tipologia ospiti
0,421472,2022-12-31,189440,BASILIADE,BASILIADE,VIA RAFFAELE PAOLUCCI N 10,5452,L'AQUILA,https://service.opdm.openpolis.io/v1/areas/5452,1.0,...,22.77,50,50.0,50.0,NaN,2,PROCEDURA NEGOZIATA SENZA PREVIA PUBBLICAZIONE...,ATTIVO,CAS ADULTI,NUCLEI FAMILIARI E MINORI
1,421474,2022-12-31,197309,IL CENACOLO DEGLI ANGELI LA RONDINE,IL CENACOLO DEGLI ANGELI LA RONDINE,VIA COLLEVERNESCO N 47 S ELIA (AQ),5452,L'AQUILA,https://service.opdm.openpolis.io/v1/areas/5452,1.0,...,22.74,36,36.0,36.0,NaN,2,PROCEDURA NEGOZIATA SENZA PREVIA PUBBLICAZIONE...,ATTIVO,CAS ADULTI,UOMINI
2,421475,2022-12-31,216582,CENTRO-a0e33f0924,L'APE,L'AQUILA PETTINO,5452,L'AQUILA,https://service.opdm.openpolis.io/v1/areas/5452,1.0,...,22.69,26,26.0,26.0,NaN,2,PROCEDURA NEGOZIATA SENZA PREVIA PUBBLICAZIONE...,ATTIVO,CAS ADULTI,UOMINI
3,421476,2022-12-31,189446,GESTIONE ORIZZONTI PARK HOTEL DA KATIA,GESTIONE ORIZZONTI PARK HOTEL DA KATIA,VIA ROMA LOCALITÀ LE COSTE SS 158 ALFEDENA,5406,ALFEDENA,https://service.opdm.openpolis.io/v1/areas/5406,5.0,...,26.20,50,50.0,50.0,NaN,2,PROCEDURA APERTA,ATTIVO,CAS ADULTI,UOMINI
4,421478,2022-12-31,189443,EDIL SAM,EDIL SAM CASTEL DI SANGRO,VIA CANAPINI CASTEL DI SANGRO,5431,CASTEL DI SANGRO,https://service.opdm.openpolis.io/v1/areas/5431,5.0,...,25.93,14,14.0,14.0,NaN,1,PROCEDURA APERTA,ATTIVO,CAS ADULTI,UOMINI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41390,428714,2023-12-31,222447,CENTRO-4805b6ee40,LIBERITUTTI --0,NaN,4025,GENOVA,https://service.opdm.openpolis.io/v1/areas/4025,1.0,...,25.03,2,2.0,2.0,NaN,1,AFFIDAMENTO DIRETTO,ATTIVO,CAS ADULTI,UOMINI
41391,433024,2023-12-31,196943,CPSA HOTSPOT,CPSA\HOTSPOT LAMPEDUSA,CONTRADA IMBRIACOLA LAMPEDUSA (AG),7421,LAMPEDUSA E LINOSA,https://service.opdm.openpolis.io/v1/areas/7421,5.0,...,28.41,763,640.0,640.0,NaN,4,NaN,ATTIVO,HOTSPOT,UOMINI DONNE MSNA NUCLEI FAMILIARI
41392,433025,2023-12-31,219832,HOT SPOT,HOT SPOT,VIALE DELLE MEDAGLIE D'ORO DI LUNGA NAVIGAZIONE,7552,POZZALLO,https://service.opdm.openpolis.io/v1/areas/7552,5.0,...,NaN,280,234.0,234.0,NaN,3,NaN,ATTIVO,HOTSPOT,UOMINI DONNE MSNA NUCLEI FAMILIARI
41393,433026,2023-12-31,219831,CPSI EX CASERMA GASPARRO,CPSI EX CASERMA GASPARRO,VIA POLVERIERA 1 BISCONTE,7341,MESSINA,https://service.opdm.openpolis.io/v1/areas/7341,1.0,...,NaN,76,200.0,200.0,NaN,3,NaN,ATTIVO,HOTSPOT,UOMINI DONNE MSNA NUCLEI FAMILIARI


In [5]:
# Create dataframes where each dataframe represents the CAS distribution by each year

cas_by_year = {
    year: cas_df[cas_df["Data rilevazione...2"] == f"{year}-12-31"].copy()
    for year in range(2018, 2024)
}

cas_data_2018 = cas_by_year[2018]
cas_data_2019 = cas_by_year[2019]
cas_data_2020 = cas_by_year[2020]
cas_data_2021 = cas_by_year[2021]
cas_data_2022 = cas_by_year[2022]
cas_data_2023 = cas_by_year[2023]

In [6]:
# Calculate the number of refugees in a commune every year by 
# summing the number of refugees in each center after grouping by Communes 

cas_comune_2018 = (cas_data_2018
    .groupby(["comune_id", "Comune"], as_index=False)
    .agg(
        inhabitants_comune=("Abitanti comune", "first"),
        total_refugees_CAS_2018=("Presenze giornaliere", "sum"),
        total_centers_CAS_2018=("id_centro", "nunique"))) 

cas_comune_2019 = (cas_data_2019
    .groupby(["comune_id", "Comune"], as_index=False)
    .agg(
        inhabitants_comune=("Abitanti comune", "first"),
        total_refugees_CAS_2019=("Presenze giornaliere", "sum"),
        total_centers_CAS_2019=("id_centro", "nunique"))) 


cas_comune_2020 = (cas_data_2020
    .groupby(["comune_id", "Comune"], as_index=False)
    .agg(
        inhabitants_comune=("Abitanti comune", "first"),
        total_refugees_CAS_2020=("Presenze giornaliere", "sum"),
        total_centers_CAS_2020=("id_centro", "nunique"))) 


cas_comune_2021 = (cas_data_2021
    .groupby(["comune_id", "Comune"], as_index=False)
    .agg(
        inhabitants_comune=("Abitanti comune", "first"),
        total_refugees_CAS_2021=("Presenze giornaliere", "sum"),
        total_centers_CAS_2021=("id_centro", "nunique"))) 


cas_comune_2022 = (cas_data_2022
    .groupby(["comune_id", "Comune"], as_index=False)
    .agg(
        inhabitants_comune=("Abitanti comune", "first"),
        total_refugees_CAS_2022=("Presenze giornaliere", "sum"),
        total_centers_CAS_2022=("id_centro", "nunique"))) 

cas_comune_2023 = (cas_data_2023
    .groupby(["comune_id", "Comune"], as_index=False)
    .agg(
        inhabitants_comune=("Abitanti comune", "first"),
        total_refugees_CAS_2023=("Presenze giornaliere", "sum"),
        total_centers_CAS_2023=("id_centro", "nunique"))) 

In [7]:
# Calculating the number of refugees per 1000 inhabitants in every commune from 2018 to 2023

cas_comune_2018["refugees_per_1000_inhabitants_2018"] = (
    cas_comune_2018["total_refugees_CAS_2018"] /
    cas_comune_2018["inhabitants_comune"] * 1000) 

cas_comune_2019["refugees_per_1000_inhabitants_2019"] = (
    cas_comune_2019["total_refugees_CAS_2019"] /
    cas_comune_2019["inhabitants_comune"] * 1000) 

cas_comune_2020["refugees_per_1000_inhabitants_2020"] = (
    cas_comune_2020["total_refugees_CAS_2020"] /
    cas_comune_2020["inhabitants_comune"] * 1000) 

cas_comune_2021["refugees_per_1000_inhabitants_2021"] = (
    cas_comune_2021["total_refugees_CAS_2021"] /
    cas_comune_2021["inhabitants_comune"] * 1000) 

cas_comune_2022["refugees_per_1000_inhabitants_2022"] = (
    cas_comune_2022["total_refugees_CAS_2022"] /
    cas_comune_2022["inhabitants_comune"] * 1000) 

cas_comune_2023["refugees_per_1000_inhabitants_2023"] = (
    cas_comune_2023["total_refugees_CAS_2023"] /
    cas_comune_2023["inhabitants_comune"] * 1000)

In [8]:
# Renaming the comune column so that the datasets can be merged 

cas_comune_2018.rename(columns={"Comune": "COMUNE"}, inplace=True)
cas_comune_2019.rename(columns={"Comune": "COMUNE"}, inplace=True) 
cas_comune_2020.rename(columns={"Comune": "COMUNE"}, inplace=True) 
cas_comune_2021.rename(columns={"Comune": "COMUNE"}, inplace=True) 
cas_comune_2022.rename(columns={"Comune": "COMUNE"}, inplace=True)

In [9]:
# Merging all the yearly datasets together 

cas_2018_2019 = merge_with_diagnostics(
    cas_comune_2018,
    cas_comune_2019,
    on=["COMUNE", "comune_id"],
    how="left",
    name="Merge 2018 and 2019 refugee data",
    suffixes=("_2018", "_2019"))

cas_2018_2019


Merge 2018 and 2019 refugee data
Left rows before merge: 2939
Right rows before merge: 2676
Rows after merge: 2939
_merge
both          2640
left_only      299
right_only       0
Name: count, dtype: int64


,comune_id,COMUNE,inhabitants_comune_2018,total_refugees_CAS_2018,total_centers_CAS_2018,refugees_per_1000_inhabitants_2018,inhabitants_comune_2019,total_refugees_CAS_2019,total_centers_CAS_2019,refugees_per_1000_inhabitants_2019
0,2,AIRASCA,3598,3,1,0.833797,3598.0,4.0,1.0,1.111729
1,4,ALBIANO D'IVREA,1644,9,2,5.474453,1644.0,16.0,3.0,9.732360
2,6,ALMESE,6426,7,2,1.089325,6426.0,8.0,3.0,1.244942
3,8,ALPIGNANO,16945,196,2,11.566834,16945.0,251.0,3.0,14.812629
4,13,AVIGLIANA,12611,14,4,1.110142,12611.0,14.0,4.0,1.110142
...,...,...,...,...,...,...,...,...,...,...
2934,8362,RIVA DEL PO,7786,30,3,3.853070,7786.0,31.0,3.0,3.981505
2935,8363,TRESIGNANA,6990,14,2,2.002861,6990.0,20.0,4.0,2.861230
2936,8364,BARBERINO TAVARNELLE,12101,16,1,1.322205,12101.0,0.0,1.0,0.000000
2937,8365,SASSOCORVARO AUDITORE,4883,14,2,2.867090,4883.0,14.0,2.0,2.867090


In [10]:
cas_2018_2020 = merge_with_diagnostics(
    cas_2018_2019,
    cas_comune_2020,
    on=["COMUNE", "comune_id"],
    how="left",
    name="Merge 2018-2019 and 2020 refugee data",
    suffixes=("", "_2020"))

cas_2018_2020


Merge 2018-2019 and 2020 refugee data
Left rows before merge: 2939
Right rows before merge: 1869
Rows after merge: 2939
_merge
both          1834
left_only     1105
right_only       0
Name: count, dtype: int64


,comune_id,COMUNE,inhabitants_comune_2018,total_refugees_CAS_2018,total_centers_CAS_2018,refugees_per_1000_inhabitants_2018,inhabitants_comune_2019,total_refugees_CAS_2019,total_centers_CAS_2019,refugees_per_1000_inhabitants_2019,inhabitants_comune,total_refugees_CAS_2020,total_centers_CAS_2020,refugees_per_1000_inhabitants_2020
0,2,AIRASCA,3598,3,1,0.833797,3598.0,4.0,1.0,1.111729,NaN,NaN,NaN,NaN
1,4,ALBIANO D'IVREA,1644,9,2,5.474453,1644.0,16.0,3.0,9.732360,1644.0,14.0,3.0,8.515815
2,6,ALMESE,6426,7,2,1.089325,6426.0,8.0,3.0,1.244942,6426.0,7.0,2.0,1.089325
3,8,ALPIGNANO,16945,196,2,11.566834,16945.0,251.0,3.0,14.812629,16945.0,169.0,3.0,9.973443
4,13,AVIGLIANA,12611,14,4,1.110142,12611.0,14.0,4.0,1.110142,12611.0,12.0,3.0,0.951550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2934,8362,RIVA DEL PO,7786,30,3,3.853070,7786.0,31.0,3.0,3.981505,7786.0,26.0,3.0,3.339327
2935,8363,TRESIGNANA,6990,14,2,2.002861,6990.0,20.0,4.0,2.861230,6990.0,25.0,4.0,3.576538
2936,8364,BARBERINO TAVARNELLE,12101,16,1,1.322205,12101.0,0.0,1.0,0.000000,NaN,NaN,NaN,NaN
2937,8365,SASSOCORVARO AUDITORE,4883,14,2,2.867090,4883.0,14.0,2.0,2.867090,4883.0,14.0,2.0,2.867090


In [11]:
cas_2018_2021 = merge_with_diagnostics(
    cas_2018_2020,
    cas_comune_2021,
    on=["COMUNE", "comune_id"],
    how="left",
    name="Merge 2018-2020 and 2021 refugee data",
    suffixes=("", "_2021"))

cas_2018_2021


Merge 2018-2020 and 2021 refugee data
Left rows before merge: 2939
Right rows before merge: 1652
Rows after merge: 2939
_merge
both          1576
left_only     1363
right_only       0
Name: count, dtype: int64


,comune_id,COMUNE,inhabitants_comune_2018,total_refugees_CAS_2018,total_centers_CAS_2018,refugees_per_1000_inhabitants_2018,inhabitants_comune_2019,total_refugees_CAS_2019,total_centers_CAS_2019,refugees_per_1000_inhabitants_2019,inhabitants_comune,total_refugees_CAS_2020,total_centers_CAS_2020,refugees_per_1000_inhabitants_2020,inhabitants_comune_2021,total_refugees_CAS_2021,total_centers_CAS_2021,refugees_per_1000_inhabitants_2021
0,2,AIRASCA,3598,3,1,0.833797,3598.0,4.0,1.0,1.111729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,ALBIANO D'IVREA,1644,9,2,5.474453,1644.0,16.0,3.0,9.732360,1644.0,14.0,3.0,8.515815,1644.0,0.0,3.0,0.000000
2,6,ALMESE,6426,7,2,1.089325,6426.0,8.0,3.0,1.244942,6426.0,7.0,2.0,1.089325,6426.0,0.0,2.0,0.000000
3,8,ALPIGNANO,16945,196,2,11.566834,16945.0,251.0,3.0,14.812629,16945.0,169.0,3.0,9.973443,16945.0,234.0,3.0,13.809383
4,13,AVIGLIANA,12611,14,4,1.110142,12611.0,14.0,4.0,1.110142,12611.0,12.0,3.0,0.951550,12611.0,3.0,4.0,0.237888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2934,8362,RIVA DEL PO,7786,30,3,3.853070,7786.0,31.0,3.0,3.981505,7786.0,26.0,3.0,3.339327,7786.0,6.0,3.0,0.770614
2935,8363,TRESIGNANA,6990,14,2,2.002861,6990.0,20.0,4.0,2.861230,6990.0,25.0,4.0,3.576538,6990.0,27.0,4.0,3.862661
2936,8364,BARBERINO TAVARNELLE,12101,16,1,1.322205,12101.0,0.0,1.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2937,8365,SASSOCORVARO AUDITORE,4883,14,2,2.867090,4883.0,14.0,2.0,2.867090,4883.0,14.0,2.0,2.867090,4883.0,12.0,2.0,2.457506


In [12]:
cas_2018_2022 = merge_with_diagnostics(
    cas_2018_2021,
    cas_comune_2022,
    on=["COMUNE", "comune_id"],
    how="left",
    name="Merge 2018-2021 and 2022 refugee data",
    suffixes=("", "_2022"))

cas_2018_2022


Merge 2018-2021 and 2022 refugee data
Left rows before merge: 2939
Right rows before merge: 1610
Rows after merge: 2939
_merge
left_only     1531
both          1408
right_only       0
Name: count, dtype: int64


,comune_id,COMUNE,inhabitants_comune_2018,total_refugees_CAS_2018,total_centers_CAS_2018,refugees_per_1000_inhabitants_2018,inhabitants_comune_2019,total_refugees_CAS_2019,total_centers_CAS_2019,refugees_per_1000_inhabitants_2019,...,total_centers_CAS_2020,refugees_per_1000_inhabitants_2020,inhabitants_comune_2021,total_refugees_CAS_2021,total_centers_CAS_2021,refugees_per_1000_inhabitants_2021,inhabitants_comune_2022,total_refugees_CAS_2022,total_centers_CAS_2022,refugees_per_1000_inhabitants_2022
0,2,AIRASCA,3598,3,1,0.833797,3598.0,4.0,1.0,1.111729,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,ALBIANO D'IVREA,1644,9,2,5.474453,1644.0,16.0,3.0,9.732360,...,3.0,8.515815,1644.0,0.0,3.0,0.000000,1644.0,8.0,1.0,4.866180
2,6,ALMESE,6426,7,2,1.089325,6426.0,8.0,3.0,1.244942,...,2.0,1.089325,6426.0,0.0,2.0,0.000000,NaN,NaN,NaN,NaN
3,8,ALPIGNANO,16945,196,2,11.566834,16945.0,251.0,3.0,14.812629,...,3.0,9.973443,16945.0,234.0,3.0,13.809383,16945.0,271.0,1.0,15.992918
4,13,AVIGLIANA,12611,14,4,1.110142,12611.0,14.0,4.0,1.110142,...,3.0,0.951550,12611.0,3.0,4.0,0.237888,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2934,8362,RIVA DEL PO,7786,30,3,3.853070,7786.0,31.0,3.0,3.981505,...,3.0,3.339327,7786.0,6.0,3.0,0.770614,7786.0,14.0,2.0,1.798099
2935,8363,TRESIGNANA,6990,14,2,2.002861,6990.0,20.0,4.0,2.861230,...,4.0,3.576538,6990.0,27.0,4.0,3.862661,6990.0,32.0,4.0,4.577969
2936,8364,BARBERINO TAVARNELLE,12101,16,1,1.322205,12101.0,0.0,1.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,12101.0,14.0,4.0,1.156929
2937,8365,SASSOCORVARO AUDITORE,4883,14,2,2.867090,4883.0,14.0,2.0,2.867090,...,2.0,2.867090,4883.0,12.0,2.0,2.457506,4883.0,14.0,2.0,2.867090


In [13]:
# Calculating the mean number of refugees per 1000 inhabitants over 2018 to 2022 

rate_cols = ["refugees_per_1000_inhabitants_2018",
    "refugees_per_1000_inhabitants_2019",
    "refugees_per_1000_inhabitants_2020",
    "refugees_per_1000_inhabitants_2021",
    "refugees_per_1000_inhabitants_2022"]

cas_2018_2022["refugees_per_1000_inhabitants_mean"] = (
    cas_2018_2022[rate_cols].mean(axis=1))

In [14]:
cas_2018_2022.to_csv(DATA_DIR/"cas_refugees_by_commune_2018_2022.csv", index=False)